# GDP forecasting with a growth target

This self-contained example forecasts annual GDP using the package's bundled data, then reconciles the results to a 4% terminal growth target. No external data download is required.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sktime.forecasting.compose import make_reduction

from macroframe_forecast import MFF
from macroframe_forecast.examples import generate_example_GDP_df

The bundled data contains annual GDP through 2024 and six missing values for 2025-2030. The forecast model uses a direct autoregression, and reconciliation imposes the terminal growth constraint.

In [ ]:
data = generate_example_GDP_df()
model = MFF(
    data,
    forecaster=make_reduction(
        LinearRegression(), strategy="direct", window_length=6,
    ),
    equality_constraints=["GDP_2030 - 1.04 * GDP_2029"],
    parallelize=False,
)
forecasts = model.fit()

growth_2030 = 100 * (forecasts.loc[2030, "GDP"] / forecasts.loc[2029, "GDP"] - 1)
print(f"Reconciled GDP growth in 2030: {growth_2030:.4f}%")
np.testing.assert_allclose(growth_2030, 4.0, rtol=0, atol=1e-8)

In [ ]:
comparison = pd.DataFrame({
    "First step": model.df1["GDP"],
    "Reconciled": forecasts["GDP"],
    "Historical": data["GDP"],
})
growth = comparison.pct_change(fill_method=None) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax, values, title, ylabel in zip(
    axes,
    [comparison, growth],
    ["GDP levels", "Annual GDP growth"],
    ["Trillion USD", "Percent"],
):
    ax.plot(values["First step"].loc[2024:], "--o", markersize=3, label="First step")
    ax.plot(values["Reconciled"].loc[2024:], "-.", label="Reconciled")
    ax.plot(values["Historical"].loc[2021:2024], color="red", label="Historical")
    ax.set(title=title, xlabel="Year", ylabel=ylabel, xlim=(2021, 2031))
    ax.set_xticks([2022, 2024, 2026, 2028, 2030])
axes[1].plot(2030, 4, "v", color="black", label="2030 target: 4%")
for ax in axes:
    ax.legend(fontsize=8)
plt.show()